# 0825_peace_017_type_expert_typewise_meta_removal

타입마다 `meta_feat1~4`가 실제 도움이 되는지 다를 수 있다는 가정으로, `0825_peace_004_type_expert_walk_forward`를 기준으로 타입별 `with_meta` / `without_meta`를 Walk-forward에서 비교하고 Validation까지 재검증합니다.

- 기준 실험은 `0825_peace_004_type_expert_walk_forward`입니다.
- 데이터 분할, XGBoost 파라미터, 임계값 선택 규칙은 004와 동일합니다.
- 차이는 이 노트북의 피처 처리만입니다.
- 실행 로그는 `docs/peace/0825_peace_017_type_expert_typewise_meta_removal.log`에 저장합니다.
- 이번 후보 비교 단계에서는 80~100% Test를 추론하지 않습니다.


## 1. 설정, 경로 탐색과 실행 로그

저장소 루트 또는 `notebooks/` 어디서 실행해도 같은 원본 파일과 로그 경로를 사용합니다.


In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_017_type_expert_typewise_meta_removal"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80
BASELINE_004_WALK = {"fixed_0.5": {"mean_pr_auc": 0.06456566580859706, "mean_recall": 0.08378869542899249, "min_recall": 0.039473684210526314, "recall_99_folds": 0, "mean_false_call_reduction": 0.997159476859665}, "global_threshold": {"mean_pr_auc": 0.06456566580859706, "mean_recall": 0.9692982456140351, "min_recall": 0.9078947368421053, "recall_99_folds": 2, "mean_false_call_reduction": 0.16944884398683502}, "type_specific_thresholds": {"mean_pr_auc": 0.06456566580859706, "mean_recall": 0.9236895920783553, "min_recall": 0.7894736842105263, "recall_99_folds": 1, "mean_false_call_reduction": 0.33363103162699187}}
BASELINE_004_VALIDATION = {"fixed_0.5": {"pr_auc": 0.4496564930298315, "recall": 0.42296918767507, "false_call_reduction": 0.995809384231377, "tp": 151, "fn": 206, "fp": 183, "tn": 43486}, "global_threshold": {"pr_auc": 0.4496564930298315, "recall": 0.9915966386554622, "false_call_reduction": 0.6993977421053836, "tp": 354, "fn": 3, "fp": 13127, "tn": 30542}, "type_specific_thresholds": {"pr_auc": 0.4496564930298315, "recall": 0.9943977591036415, "false_call_reduction": 0.5644965536192722, "tp": 355, "fn": 2, "fp": 19018, "tn": 24651}}

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


def safe_float(value):
    if isinstance(value, (np.floating, float)):
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    return value


def to_builtin_dict(value):
    if isinstance(value, dict):
        return {str(k): to_builtin_dict(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_builtin_dict(v) for v in value]
    if isinstance(value, tuple):
        return [to_builtin_dict(v) for v in value]
    return safe_float(value)


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 16:30:32,237 | INFO | experiment=0825_peace_017_type_expert_typewise_meta_removal


2026-08-25 16:30:32,238 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99


2026-08-25 16:30:32,238 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 16:30:32,239 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 16:30:32,239 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 16:30:32,239 | INFO | log_file=docs/peace/0825_peace_017_type_expert_typewise_meta_removal.log


log saved to: docs/peace/0825_peace_017_type_expert_typewise_meta_removal.log


## 2. 원본 데이터와 매핑 검증

첫 번째 익명 인덱스 열은 `record_id`로 이름만 바꾸며 원본 파일은 수정하지 않습니다.


In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)

inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
mapped_feature_columns_by_type = {}
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    mapped_feature_columns_by_type[inspection_type] = mapped_columns
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 16:30:36,639 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 16:30:36,644 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 3. 시간순 분할과 평가 함수

004와 같은 시간순 70/10/20 분할을 유지하되, 이번 노트북은 70~80% Validation까지만 사용하고 마지막 20% Test는 건드리지 않습니다.


In [3]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask].copy()
validation_df = raw_df.loc[validation_mask].copy()
test_df = raw_df.loc[test_mask].copy()
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test_holdout_unused",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))


def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(np.r_[sorted_probability[:-1] != sorted_probability[1:], True])

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort((thresholds[feasible], recall[feasible], false_call_reduction[feasible]))[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")


def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def robust_stat_frame(reference_df):
    medians = reference_df.median(axis=0)
    abs_dev = reference_df.sub(medians, axis=1).abs()
    mad = abs_dev.median(axis=0)
    raw_scale = 1.4826 * mad
    scale = raw_scale.where(raw_scale > 0, 1.0).fillna(1.0)
    return pd.DataFrame({"median": medians, "mad": mad, "scale": scale})


WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end].copy(),
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ].copy(),
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ].copy(),
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(["fold", "segment"])
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test_holdout_unused,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


2026-08-25 16:30:37,036 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test_holdout_unused', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 16:30:37,054 | INFO | threshold_selector_unit_test=PASS


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 16:30:37,236 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

## 4. 타입별 Meta Feature ablation

각 타입에서 `with_meta`와 `without_meta`를 동일한 Walk-forward fold로 비교하고, Walk-forward 결과만으로 타입별 구성을 선택합니다.

In [4]:
VARIANT_NAMES = ["with_meta", "without_meta"]
feature_columns_by_type_variant = {
    inspection_type: {
        "with_meta": feature_columns_by_type[inspection_type],
        "without_meta": mapped_feature_columns_by_type[inspection_type],
    }
    for inspection_type in inspection_types
}
variant_feature_summary = pd.DataFrame(
    [
        {
            "inspection_type": inspection_type,
            "variant": variant_name,
            "raw_features": len(feature_columns),
        }
        for inspection_type in inspection_types
        for variant_name, feature_columns in feature_columns_by_type_variant[inspection_type].items()
    ]
).set_index(["inspection_type", "variant"])
display(variant_feature_summary)


def fit_variant_probabilities(train_frame, calibration_frame, evaluation_frame, fold_name):
    probability_store = {"calibration": {}, "evaluation": {}}
    variant_metric_rows = []
    training_rows = []

    for inspection_type in inspection_types:
        type_train = train_frame.loc[train_frame[TYPE_COLUMN] == inspection_type].copy()
        type_calibration = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type].copy()
        type_evaluation = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type].copy()
        y_train = type_train[TARGET].astype("int8")

        assert len(type_train) > 0
        assert len(type_calibration) > 0
        assert len(type_evaluation) > 0
        assert y_train.nunique() == 2
        assert type_calibration[TARGET].nunique() == 2
        assert type_train[TIME_COLUMN].max() < type_calibration[TIME_COLUMN].min()
        assert type_calibration[TIME_COLUMN].max() < type_evaluation[TIME_COLUMN].min()

        for variant_name in VARIANT_NAMES:
            feature_columns = feature_columns_by_type_variant[inspection_type][variant_name]
            preprocessor = make_preprocessor(feature_columns)
            X_train = preprocessor.fit_transform(type_train[feature_columns])
            X_calibration = preprocessor.transform(type_calibration[feature_columns])
            X_evaluation = preprocessor.transform(type_evaluation[feature_columns])

            model = XGBClassifier(**XGB_PARAMS)
            model.fit(X_train, y_train, verbose=False)
            calibration_probability = pd.Series(
                model.predict_proba(X_calibration)[:, 1],
                index=type_calibration.index,
                dtype="float64",
            )
            evaluation_probability = pd.Series(
                model.predict_proba(X_evaluation)[:, 1],
                index=type_evaluation.index,
                dtype="float64",
            )
            probability_store["calibration"][(inspection_type, variant_name)] = calibration_probability
            probability_store["evaluation"][(inspection_type, variant_name)] = evaluation_probability

            selection = select_threshold(type_calibration[TARGET], calibration_probability, min_recall=MIN_RECALL)
            thresholded_metrics = evaluate_predictions(
                type_evaluation[TARGET],
                (evaluation_probability >= selection["threshold"]).astype("int8"),
                evaluation_probability,
            )
            base_metrics = evaluate_probabilities(type_evaluation[TARGET], evaluation_probability, DECISION_THRESHOLD)
            variant_metric_rows.append(
                {
                    "fold": fold_name,
                    "inspection_type": inspection_type,
                    "variant": variant_name,
                    "calibration_threshold": selection["threshold"],
                    "pr_auc": base_metrics["pr_auc"],
                    "roc_auc": base_metrics["roc_auc"],
                    "fixed_recall": base_metrics["recall"],
                    "fixed_false_call_reduction": base_metrics["false_call_reduction"],
                    "thresholded_recall": thresholded_metrics["recall"],
                    "thresholded_false_call_reduction": thresholded_metrics["false_call_reduction"],
                    "tp": thresholded_metrics["tp"],
                    "fn": thresholded_metrics["fn"],
                    "fp": thresholded_metrics["fp"],
                    "tn": thresholded_metrics["tn"],
                    "raw_features": len(feature_columns),
                    "encoded_features": X_train.shape[1],
                }
            )
            training_rows.append(
                {
                    "fold": fold_name,
                    "inspection_type": inspection_type,
                    "variant": variant_name,
                    "train_rows": len(type_train),
                    "train_positive": int(y_train.sum()),
                    "calibration_rows": len(type_calibration),
                    "evaluation_rows": len(type_evaluation),
                    "raw_features": len(feature_columns),
                    "encoded_features": X_train.shape[1],
                }
            )
            logger.info(
                "walk_forward_variant_fit_done fold=%s type=%d variant=%s raw_features=%d",
                fold_name,
                inspection_type,
                variant_name,
                len(feature_columns),
            )
            del preprocessor, model, X_train, X_calibration, X_evaluation
            gc.collect()

    return probability_store, variant_metric_rows, training_rows


probability_store_by_fold = {}
variant_metric_rows = []
walk_forward_training_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    segments = walk_forward_segments[fold_name]
    probability_store, fold_variant_metric_rows, training_rows = fit_variant_probabilities(
        segments["train"], segments["calibration"], segments["evaluation"], fold_name
    )
    probability_store_by_fold[fold_name] = probability_store
    variant_metric_rows.extend(fold_variant_metric_rows)
    walk_forward_training_rows.extend(training_rows)

variant_metrics = pd.DataFrame(variant_metric_rows)
variant_summary = (
    variant_metrics.groupby(["inspection_type", "variant"])
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_roc_auc=("roc_auc", "mean"),
        mean_fixed_recall=("fixed_recall", "mean"),
        mean_thresholded_recall=("thresholded_recall", "mean"),
        mean_thresholded_false_call_reduction=("thresholded_false_call_reduction", "mean"),
        mean_encoded_features=("encoded_features", "mean"),
    )
    .reset_index()
)
display(variant_metrics)
display(variant_summary)

selected_variant_by_type = {}
selection_rows = []
for inspection_type in inspection_types:
    subset = variant_summary.loc[variant_summary["inspection_type"] == inspection_type].copy()
    subset = subset.sort_values(
        by=[
            "mean_pr_auc",
            "mean_thresholded_recall",
            "mean_thresholded_false_call_reduction",
            "mean_roc_auc",
        ],
        ascending=False,
        kind="stable",
    )
    best = subset.iloc[0]
    selected_variant_by_type[inspection_type] = best["variant"]
    selection_rows.append(
        {
            "inspection_type": inspection_type,
            "selected_variant": best["variant"],
            "mean_pr_auc": best["mean_pr_auc"],
            "mean_thresholded_recall": best["mean_thresholded_recall"],
            "mean_thresholded_false_call_reduction": best["mean_thresholded_false_call_reduction"],
            "mean_encoded_features": best["mean_encoded_features"],
        }
    )

selected_variant_summary = pd.DataFrame(selection_rows).set_index("inspection_type")
display(selected_variant_summary)
logger.info("selected_variant_by_type=%s", selected_variant_by_type)

walk_forward_threshold_rows = []
walk_forward_metric_rows = []
walk_forward_type_evaluation_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    segments = walk_forward_segments[fold_name]
    calibration_frame = segments["calibration"]
    evaluation_frame = segments["evaluation"]
    calibration_probability = pd.Series(np.nan, index=calibration_frame.index, dtype="float64")
    evaluation_probability = pd.Series(np.nan, index=evaluation_frame.index, dtype="float64")

    for inspection_type in inspection_types:
        variant_name = selected_variant_by_type[inspection_type]
        calibration_probability.loc[
            calibration_frame.index.intersection(
                probability_store_by_fold[fold_name]["calibration"][(inspection_type, variant_name)].index
            )
        ] = probability_store_by_fold[fold_name]["calibration"][(inspection_type, variant_name)]
        evaluation_probability.loc[
            evaluation_frame.index.intersection(
                probability_store_by_fold[fold_name]["evaluation"][(inspection_type, variant_name)].index
            )
        ] = probability_store_by_fold[fold_name]["evaluation"][(inspection_type, variant_name)]

    assert calibration_probability.notna().all()
    assert evaluation_probability.notna().all()

    global_selection = select_threshold(calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL)
    walk_forward_threshold_rows.append({"fold": fold_name, "scope": "global", **global_selection})

    type_evaluation_prediction = pd.Series(np.nan, index=evaluation_frame.index, dtype="float64")
    for inspection_type in inspection_types:
        variant_name = selected_variant_by_type[inspection_type]
        type_calibration = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type]
        type_calibration_probability = probability_store_by_fold[fold_name]["calibration"][(inspection_type, variant_name)]
        selection = select_threshold(type_calibration[TARGET], type_calibration_probability, min_recall=MIN_RECALL)
        threshold = selection["threshold"]
        walk_forward_threshold_rows.append({"fold": fold_name, "scope": f"type_{inspection_type}", **selection})

        type_evaluation = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type]
        type_evaluation_probability = probability_store_by_fold[fold_name]["evaluation"][(inspection_type, variant_name)]
        type_prediction = (type_evaluation_probability >= threshold).astype("int8")
        type_evaluation_prediction.loc[type_evaluation.index] = type_prediction
        type_metrics = evaluate_predictions(type_evaluation[TARGET], type_prediction, type_evaluation_probability)
        type_metrics.update({"fold": fold_name, "inspection_type": inspection_type, "threshold": threshold})
        walk_forward_type_evaluation_rows.append(type_metrics)

    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD),
        "global_threshold": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, global_selection["threshold"]),
        "type_specific_thresholds": evaluate_predictions(evaluation_frame[TARGET], type_evaluation_prediction, evaluation_probability),
    }
    for strategy, metrics in strategy_metrics.items():
        walk_forward_metric_rows.append({"fold": fold_name, "strategy": strategy, **metrics})

walk_forward_threshold_summary = pd.DataFrame(walk_forward_threshold_rows).set_index(["fold", "scope"])
walk_forward_evaluation_metrics = pd.DataFrame(walk_forward_metric_rows).set_index(["fold", "strategy"])
walk_forward_type_evaluation = pd.DataFrame(walk_forward_type_evaluation_rows).set_index(["fold", "inspection_type"])
walk_forward_training_summary = pd.DataFrame(walk_forward_training_rows).set_index(["fold", "inspection_type", "variant"])

display(walk_forward_threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_evaluation_metrics[["positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(walk_forward_type_evaluation[["threshold", "positive_samples", "pr_auc", "recall", "false_call_reduction", "tp", "fn"]])

walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index()
    .groupby("strategy")
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"),
        min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"),
        total_fn=("fn", "sum"),
    )
)
display(walk_forward_strategy_summary)
display(walk_forward_training_summary)
logger.info("walk_forward_strategy_summary=%s", walk_forward_strategy_summary.to_dict(orient="index"))

pooled_probability = pd.Series(np.nan, index=validation_df.index, dtype="float64")
type_metric_rows = []
training_rows = []

for inspection_type in inspection_types:
    variant_name = selected_variant_by_type[inspection_type]
    feature_columns = feature_columns_by_type_variant[inspection_type][variant_name]
    type_train = train_df.loc[train_df[TYPE_COLUMN] == inspection_type].copy()
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type].copy()
    y_train = type_train[TARGET].astype("int8")

    assert len(type_train) > 0 and len(type_validation) > 0
    assert y_train.nunique() == 2
    assert type_train[TIME_COLUMN].max() < type_validation[TIME_COLUMN].min()

    preprocessor = make_preprocessor(feature_columns)
    X_train = preprocessor.fit_transform(type_train[feature_columns])
    X_validation = preprocessor.transform(type_validation[feature_columns])

    model = XGBClassifier(**XGB_PARAMS)
    model.fit(X_train, y_train, verbose=False)
    probability = model.predict_proba(X_validation)[:, 1]
    pooled_probability.loc[type_validation.index] = probability

    metrics = evaluate_probabilities(type_validation[TARGET], probability)
    metrics["inspection_type"] = inspection_type
    type_metric_rows.append(metrics)
    training_rows.append(
        {
            "inspection_type": inspection_type,
            "selected_variant": variant_name,
            "train_rows": len(type_train),
            "train_positive": int(y_train.sum()),
            "validation_rows": len(type_validation),
            "validation_positive": int(type_validation[TARGET].sum()),
            "raw_features": len(feature_columns),
            "encoded_features": X_train.shape[1],
            "trees": model.n_estimators,
        }
    )
    logger.info("validation_fit_done type=%d variant=%s", inspection_type, variant_name)
    del X_train, X_validation, probability, model, preprocessor
    gc.collect()

assert pooled_probability.notna().all()
pooled_metrics = pd.Series(evaluate_probabilities(validation_df[TARGET], pooled_probability), name="type_expert_validation")
type_metrics = pd.DataFrame(type_metric_rows).set_index("inspection_type")
training_summary = pd.DataFrame(training_rows).set_index("inspection_type")
count_columns = ["rows", "positive_samples", "tn", "fp", "fn", "tp"]
type_metrics[count_columns] = type_metrics[count_columns].astype("int64")

display(pooled_metrics)
display(type_metrics[["rows", "positive_samples", "pr_auc", "roc_auc", "accuracy", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(training_summary)

global_threshold_selection = select_threshold(validation_df[TARGET], pooled_probability, min_recall=MIN_RECALL)
thresholds_by_type = {}
type_threshold_rows = []
type_validation_prediction = pd.Series(np.nan, index=validation_df.index, dtype="float64")

for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_probability = pooled_probability.loc[type_validation.index]
    selection = select_threshold(type_validation[TARGET], type_probability, min_recall=MIN_RECALL)
    thresholds_by_type[inspection_type] = selection["threshold"]
    selection["inspection_type"] = inspection_type
    type_threshold_rows.append(selection)
    type_validation_prediction.loc[type_validation.index] = (type_probability >= selection["threshold"]).astype("int8")

type_threshold_selection = pd.DataFrame(type_threshold_rows).set_index("inspection_type")
global_validation_metrics = pd.Series(
    evaluate_probabilities(validation_df[TARGET], pooled_probability, global_threshold_selection["threshold"]),
    name="global_threshold",
)
type_specific_validation_metrics = pd.Series(
    evaluate_predictions(validation_df[TARGET], type_validation_prediction, pooled_probability),
    name="type_specific_thresholds",
)
validation_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": pooled_metrics,
        "global_threshold": global_validation_metrics,
        "type_specific_thresholds": type_specific_validation_metrics,
    }
).T
threshold_summary = pd.concat(
    [
        pd.DataFrame([{"scope": "global", **global_threshold_selection}]).set_index("scope"),
        type_threshold_selection.rename_axis("scope"),
    ],
    axis=0,
)
display(threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(validation_strategy_metrics[["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
logger.info("validation_strategy_metrics=%s", validation_strategy_metrics.to_dict(orient="index"))


raw_features
inspection_type variant                   
0               with_meta               48
                without_meta            44
1               with_meta               56
                without_meta            52
2               with_meta               69
                without_meta            65
3               with_meta               69
                without_meta            65
4               with_meta               25
                without_meta            21

2026-08-25 16:30:37,650 | INFO | walk_forward_variant_fit_done fold=fold_1 type=0 variant=with_meta raw_features=48


2026-08-25 16:30:37,962 | INFO | walk_forward_variant_fit_done fold=fold_1 type=0 variant=without_meta raw_features=44


2026-08-25 16:30:38,657 | INFO | walk_forward_variant_fit_done fold=fold_1 type=1 variant=with_meta raw_features=56


2026-08-25 16:30:39,517 | INFO | walk_forward_variant_fit_done fold=fold_1 type=1 variant=without_meta raw_features=52


...

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,13289,12,0.004690,0.863599,0.999097,0.000000,0.000000,1.000000,0.000000,0,12,0,13277
1,6422,224,0.697741,0.968575,0.975553,0.679144,0.566964,0.990319,0.618005,127,97,60,6138
2,7161,27,0.323275,0.938749,0.996230,0.500000,0.074074,0.999720,0.129032,2,25,2,7132
3,16252,21,0.071857,0.914731,0.998154,0.090909,0.047619,0.999384,0.062500,1,20,10,16221
4,902,73,0.080931,0.500000,0.919069,0.000000,0.000000,1.000000,0.000000,0,73,0,829


,selected_variant,train_rows,train_positive,validation_rows,validation_positive,raw_features,encoded_features,trees
inspection_type,,,,,,,,
0,with_meta,64273,111,13289,12,48,88,400
1,without_meta,38900,580,6422,224,52,52,400
2,without_meta,100470,588,7161,27,65,65,400
3,without_meta,100740,648,16252,21,65,65,400
4,with_meta,3813,13,902,73,25,53,400


,threshold,positive_samples,recall,false_call_reduction,tp,fn,fp,tn
scope,,,,,,,,
global,0.000603,357,0.991597,0.626463,354,3,16312,27357
0,0.000231,12,1.000000,0.498682,12,0,6656,6621
1,0.001392,224,0.991071,0.618748,222,2,2363,3835
2,0.000209,27,1.000000,0.348893,27,0,4645,2489
3,0.000283,21,1.000000,0.522026,21,0,7758,8473
4,0.003448,73,1.000000,0.000000,73,0,829,0


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.471722,0.643564,0.364146,0.998351,0.465116,130.0,227.0,72.0,43597.0
global_threshold,0.471722,0.021241,0.991597,0.626463,0.041591,354.0,3.0,16312.0,27357.0
type_specific_thresholds,0.471722,0.015704,0.994398,0.490462,0.030919,355.0,2.0,22251.0,21418.0


2026-08-25 16:31:10,267 | INFO | validation_strategy_metrics={'fixed_0.5': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43597.0, 'fp': 72.0, 'fn': 227.0, 'tp': 130.0, 'accuracy': 0.9932085585790215, 'precision': 0.6435643564356436, 'recall': 0.3641456582633053, 'false_call_reduction': 0.9983512331402139, 'f1': 0.46511627906976744, 'roc_auc': 0.9511612472051496, 'pr_auc': 0.471722425482273}, 'global_threshold': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 27357.0, 'fp': 16312.0, 'fn': 3.0, 'tp': 354.0, 'accuracy': 0.6294235224639986, 'precision': 0.02124084963398536, 'recall': 0.9915966386554622, 'false_call_reduction': 0.6264627080995672, 'f1': 0.041590788932620575, 'roc_auc': 0.9511612472051496, 'pr_auc': 0.471722425482273}, 'type_specific_thresholds': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 21418.0, 'fp': 22251.0, 'fn': 2.0, 'tp': 355.0, 'accuracy': 0.4945486757824922, 'precision': 0.015703795452534725, 'recall': 0.9943977591036415, 'false_call_reduction': 0.49

## 5. 원본 무결성과 결론

실행 전후 원본 해시가 같은지 확인하고, 004 대비 핵심 변화만 요약합니다.

In [5]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE

verification = pd.Series(
    {
        "dataset_sha256_unchanged": True,
        "mapping_sha256_unchanged": True,
        "test_holdout_used_for_selection": False,
        "test_holdout_predicted": False,
        "fixed_threshold": DECISION_THRESHOLD,
        "global_threshold": global_threshold_selection["threshold"],
        "type_thresholds": thresholds_by_type,
        "log_file": f"docs/peace/{LOG_PATH.name}",
    },
    name="verification",
)
display(verification)

baseline_walk_global = BASELINE_004_WALK["global_threshold"]
baseline_validation_global = BASELINE_004_VALIDATION["global_threshold"]
experiment_specific_summary = {
    "engineered_features": ENGINEERED_FEATURES if "ENGINEERED_FEATURES" in globals() else None,
}
if "selected_variant_by_type" in globals():
    experiment_specific_summary["selected_variant_by_type"] = selected_variant_by_type
if "walk_forward_feature_stats_summary" in globals():
    experiment_specific_summary["feature_stat_rows"] = int(len(walk_forward_feature_stats_summary))
if "validation_feature_stats" in globals():
    experiment_specific_summary["validation_feature_stat_rows"] = int(len(validation_feature_stats))

result_summary = {
    "experiment_id": EXPERIMENT_ID,
    "notebook_path": f"notebooks/{EXPERIMENT_ID}.ipynb",
    "log_path": f"docs/peace/{LOG_PATH.name}",
    "evaluation_scope": "walk_forward_and_validation_only",
    "baseline_004_reference": {
        "walk_forward": BASELINE_004_WALK,
        "validation": BASELINE_004_VALIDATION,
    },
    "walk_forward_strategy_summary": to_builtin_dict(walk_forward_strategy_summary.to_dict(orient="index")),
    "validation_strategy_metrics": to_builtin_dict(validation_strategy_metrics.to_dict(orient="index")),
    "threshold_summary": to_builtin_dict(threshold_summary.reset_index().to_dict(orient="records")),
    "experiment_specific_summary": to_builtin_dict(experiment_specific_summary),
}
if "selected_variant_summary" in globals():
    result_summary["selected_variant_summary"] = to_builtin_dict(selected_variant_summary.reset_index().to_dict(orient="records"))
if "variant_summary" in globals():
    result_summary["variant_summary"] = to_builtin_dict(variant_summary.to_dict(orient="records"))

logger.info("result_summary_json=%s", json.dumps(result_summary, ensure_ascii=False, sort_keys=True))
logger.info("source_integrity=PASS test_holdout_predicted=False global_threshold=%.8f", global_threshold_selection["threshold"])
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()

global_walk = walk_forward_strategy_summary.loc["global_threshold"]
global_valid = validation_strategy_metrics.loc["global_threshold"]
type_walk = walk_forward_strategy_summary.loc["type_specific_thresholds"]
type_valid = validation_strategy_metrics.loc["type_specific_thresholds"]

conclusion_lines = [
    f"# {{EXPERIMENT_ID}} 결론",
    "",
    f"- Walk-forward 공통 임계값 평균 Recall은 {{global_walk['mean_recall']:.4f}}, 평균 FCR은 {{global_walk['mean_false_call_reduction']:.4f}}이다.",
    f"- 004 대비 공통 임계값 평균 Recall 변화는 {{global_walk['mean_recall'] - baseline_walk_global['mean_recall']:+.4f}}, 평균 FCR 변화는 {{global_walk['mean_false_call_reduction'] - baseline_walk_global['mean_false_call_reduction']:+.4f}}이다.",
    f"- Validation 공통 임계값 Recall은 {{global_valid['recall']:.4f}}, FCR은 {{global_valid['false_call_reduction']:.4f}}이다.",
    f"- 004 대비 Validation 공통 임계값 Recall 변화는 {{global_valid['recall'] - baseline_validation_global['recall']:+.4f}}, FCR 변화는 {{global_valid['false_call_reduction'] - baseline_validation_global['false_call_reduction']:+.4f}}이다.",
    f"- 타입별 임계값 Walk-forward 평균 Recall/FCR은 {{type_walk['mean_recall']:.4f}} / {{type_walk['mean_false_call_reduction']:.4f}}이다.",
    f"- 타입별 임계값 Validation Recall/FCR은 {{type_valid['recall']:.4f}} / {{type_valid['false_call_reduction']:.4f}}이다.",
    "- 이 노트북은 최종 모델 선택 전 후보 비교용이며 80~100% Test는 추론하지 않았다.",
]
if "selected_variant_by_type" in globals():
    variant_text = ", ".join(f"type {{k}}={{v}}" for k, v in selected_variant_by_type.items())
    conclusion_lines.append(f"- 선택된 meta 제거 설정: {{variant_text}}")
if "ENGINEERED_FEATURES" in globals():
    conclusion_lines.append(f"- 추가된 파생 변수: {{', '.join(ENGINEERED_FEATURES)}}")

display(Markdown("\n".join(conclusion_lines)))


dataset_sha256_unchanged                                                        True
mapping_sha256_unchanged                                                        True
test_holdout_used_for_selection                                                False
test_holdout_predicted                                                         False
fixed_threshold                                                                  0.5
global_threshold                                                            0.000603
type_thresholds                    {0: 0.00023106035951059312, 1: 0.0013922951184...
log_file                           docs/peace/0825_peace_017_type_expert_typewise...
Name: verification, dtype: object

2026-08-25 16:31:10,470 | INFO | result_summary_json={"baseline_004_reference": {"validation": {"fixed_0.5": {"false_call_reduction": 0.995809384231377, "fn": 206, "fp": 183, "pr_auc": 0.4496564930298315, "recall": 0.42296918767507, "tn": 43486, "tp": 151}, "global_threshold": {"false_call_reduction": 0.6993977421053836, "fn": 3, "fp": 13127, "pr_auc": 0.4496564930298315, "recall": 0.9915966386554622, "tn": 30542, "tp": 354}, "type_specific_thresholds": {"false_call_reduction": 0.5644965536192722, "fn": 2, "fp": 19018, "pr_auc": 0.4496564930298315, "recall": 0.9943977591036415, "tn": 24651, "tp": 355}}, "walk_forward": {"fixed_0.5": {"mean_false_call_reduction": 0.997159476859665, "mean_pr_auc": 0.06456566580859706, "mean_recall": 0.08378869542899249, "min_recall": 0.039473684210526314, "recall_99_folds": 0}, "global_threshold": {"mean_false_call_reduction": 0.16944884398683502, "mean_pr_auc": 0.06456566580859706, "mean_recall": 0.9692982456140351, "min_recall": 0.9078947368421053, "re

2026-08-25 16:31:10,471 | INFO | source_integrity=PASS test_holdout_predicted=False global_threshold=0.00060295


2026-08-25 16:31:10,471 | INFO | experiment_complete=0825_peace_017_type_expert_typewise_meta_removal


# {EXPERIMENT_ID} 결론

- Walk-forward 공통 임계값 평균 Recall은 {global_walk['mean_recall']:.4f}, 평균 FCR은 {global_walk['mean_false_call_reduction']:.4f}이다.
- 004 대비 공통 임계값 평균 Recall 변화는 {global_walk['mean_recall'] - baseline_walk_global['mean_recall']:+.4f}, 평균 FCR 변화는 {global_walk['mean_false_call_reduction'] - baseline_walk_global['mean_false_call_reduction']:+.4f}이다.
- Validation 공통 임계값 Recall은 {global_valid['recall']:.4f}, FCR은 {global_valid['false_call_reduction']:.4f}이다.
- 004 대비 Validation 공통 임계값 Recall 변화는 {global_valid['recall'] - baseline_validation_global['recall']:+.4f}, FCR 변화는 {global_valid['false_call_reduction'] - baseline_validation_global['false_call_reduction']:+.4f}이다.
- 타입별 임계값 Walk-forward 평균 Recall/FCR은 {type_walk['mean_recall']:.4f} / {type_walk['mean_false_call_reduction']:.4f}이다.
- 타입별 임계값 Validation Recall/FCR은 {type_valid['recall']:.4f} / {type_valid['false_call_reduction']:.4f}이다.
- 이 노트북은 최종 모델 선택 전 후보 비교용이며 80~100% Test는 추론하지 않았다.
- 선택된 meta 제거 설정: {variant_text}